# 04 — Model based grading

When evaluating AI models that generate code, you need more than just checking if the response makes sense. You also need to verify that the generated code actually has valid syntax and follows the correct format. This is where code-based grading comes in.

- Format - The response should return only the requested code type (Python, JSON, or Regex) without explanations
- Valid Syntax - The generated code should actually parse correctly as the intended language
- Task Following - The response should directly address what was asked and be accurate

The first two criteria are handled by the code grader, while task following is evaluated by the model grader. Together, they provide a comprehensive evaluation.

In [1]:
# Add repo root to path
import sys
sys.path.append("../..")

In [2]:
# Get the client and model from utils
from src.utils import get_client, get_model

client = get_client()
model = get_model()

In [3]:
# Helper functions
from src.utils import chat, add_user_message, add_assistant_message

In [4]:
# Other imports
from src.prompt_evaluation import generate_dataset, run_eval

In [5]:
import json

dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

## 1. Syntax validation functions

To check if generated code has valid syntax, you can create three helper functions that attempt to parse the output:

In [6]:
# Functions to validate the output structure
import json
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


In [8]:
with open('dataset.json', 'r') as f:
    dataset = json.load(f)


results = run_eval(dataset)

Average Score: 8.0


In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:GetObjectVersion\",\n        \"s3:ListBucket\",\n        \"s3:ListBucketVersions\",\n        \"s3:GetBucketLocation\",\n        \"s3:GetBucketVersioning\",\n        \"s3:ListAllMyBuckets\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::*\",\n        \"arn:aws:s3:::*/*\"\n      ]\n    }\n  ]\n}\n",
    "test_case": {
      "task": "Create a JSON policy document that allows read-only access to all S3 buckets for a specific IAM user",
      "format": "json"
    },
    "score": 9.0,
    "reasoning": "The policy correctly provides read-only S3 access with appropriate permissions and resource coverage. However, it doesn't address the 'specific IAM user' requirement - this would typically be handled through policy attachment rather than within the policy document itself. The core functionality is sound but cou